# Time series analysis
---

Running this notebook you will:

- Plot the timeseries
- Extract the features
- Plot the univarialte and bivariate relation between the features

In [1]:
#@markdown ###Run this cell to connect your Google Drive to Colab and install packages

#@markdown After the first execution you might receive some warning and notifications, please follow these instructions:

#@markdown * Warning: This notebook was not authored by Google. Click on *'Run anyway'*.
#@markdown * Permit this notebook to access your Google Drive files? Click on *'Yes'*, and select your account.
#@markdown * Google Drive for desktop wants to access your Google Account. Click on *'Allow'*.

# Mounts user's Google Drive to Google Colab switch folder.
from google.colab import drive
drive.mount('/content/gdrive')
#%cd /content/gdrive/MyDrive/
#!git clone https://github.com/HelmholtzAI-Consultants-Munich/NMOs-Contraction.git
%cd /content/gdrive/MyDrive/NMO-colab/

# Install necessary packages
%pip install tsfresh
%pip install -U kaleido

Mounted at /content/gdrive
/content/gdrive/MyDrive/NMO-colab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.1/169.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 7.7 MB/s eta 0:00:00


In [2]:
#@markdown ## Import
#@markdown Run the next cell to import the necessary packages

import glob
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go
import kaleido
# from IPython.display import Markdown as md
import ipywidgets as widgets
from IPython.display import display
from IPython.display import display_markdown
from utils.pre_processing import process_single_ts
from utils.feature_extraction import create_feature_in_dataframe
from utils.plot_boxplot import plot_box_plots

In [3]:
#@markdown ## Custom data path

#@markdown In case you change the folder structure, you can insert here the paths to the data (extracted time series) and the figures.
#@markdown If you change something, run the cell after the modification.
data_path = 'extracted_signals' #@param {type:"string"}
figure_path = 'figures' #@param {type:"string"}
figure_path = figure_path + '/'
#@markdown ## Select excel file
excel_file = 'Datasheet_template_example.xlsx' #@param {type:"string"}


In [4]:
#@markdown ## Pre-processing
#@markdown Running this cell the you will open the time series and the info file data, and pre-process the time series.

#@markdown The pre-processing consist of the following steps:
#@markdown - polynomial interpolation in case missing values are present
#@markdown - de-trending
#@markdown - smoothing
#@markdown - scaling: pixel in micrometer and bin in seconds

#@markdown  If you want to print the complete list of all the processed data check the box below.

# Open data info file
data_info = pd.read_excel(excel_file)
# Rename columns and fill NA values
data_info.rename(columns={'File_video name': 'Video name'}, inplace=True)
data_info['Treatment during contraction'].fillna('Other', inplace=True)
# data_info['Treatment/Phenotype'].fillna('Other', inplace=True)

# Open time series files
filename_list = []
data = []
files = []

for filename in glob.glob(os.path.join(data_path, '*.npy')):
    filename_list.append(Path(filename).stem)
    with open(filename, 'r') as f:
        data.append([Path(filename).stem, np.load(filename)])

print_filenames = True #@param {type:"boolean"}
if print_filenames:
  for index, name in enumerate(filename_list):
     print(index, name)

# Delete tail corrupted data
if 'O1318_1b' in filename_list:
  element = 'O1318_1b'
  print(f'The tale of {element} has been cut beacuse the data were currupted.')
  index = filename_list.index(element)
  data[index][1] = data[index][1][:, :650]

# Pre-processing
data_processed = []

for data_i in data:
  data_processed_i, is_nan_flag = process_single_ts(data_i[1], smooth=True, de_trend=True)
  data_processed.append([data_i[0], data_processed_i])
  if is_nan_flag:
    print(f'The signal {data_i[0]} has NaNs and the missing values have been interpolated')

print(f'Number of processed files: {len(filename_list)}')

0 P1365_SMA_RISD_d54_O3_b_3
1 P1365_SMA_RISD_d54_O1_a_2
2 P1365_SMA_BRAN_d58_O2_b_2
Number of processed files: 3


In [ ]:
#@markdown ## Plot all time series
#@markdown Run this cell <b>only</b> if you want to visualize all the time series after the pre-processing step
#@markdown The plots will be saved in the figure folder (in a unique file called 'All Time Series' and also in multiple small windows for a better visualization.

#@markdown Here you can choose where to set the value for the horizontal dashed line to help you visualizing the different time series

line_of_reference = 0.5 #@param {type:"number"}
t_weak = line_of_reference
t_conversion = 0.097
plot_info = {'y_min': -2.5,
             'y_max': 2.5,
             'subplot_layout_row': 34,
             'subplot_layout_col': 2,
             'title': 'All Time Series'}

colors = ["#43C6DB", "#93FFE8", "dodgerblue", "mediumseagreen", "slateblue", "#737CA1", "#29465B", "#368BC1",
          "#AFDCEC", "#66CDAA"]

fig, axis = plt.subplots(plot_info['subplot_layout_row'], plot_info['subplot_layout_col'], figsize=(20, 70))
fig.suptitle(plot_info['title'], fontsize=16)
plt.tight_layout(pad=3.)
for ax, data_i in zip(axis.flat, data_processed):
    for i in range(data_i[1].shape[0]):
        ax.plot([t * t_conversion for t in range(len(data_i[1][i, :]))], data_i[1][i, :], color=colors[i], alpha=0.5)
    ax.set_title(data_i[0], y=2., pad=-14)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('y [\u03BCm]')
    ax.set_ylim(plot_info['y_min'], plot_info['y_max'])
    ax.axhline(y=t_weak, color='g', linestyle='--', label='std')
    ax.axhline(y=-t_weak, color='g', linestyle='--', label='std')
fig.savefig(figure_path + plot_info['title'])
# plot_all_signals = True #@param {type:"boolean"}
# if plot_all_signals == False:
#   plt.close()

# Save figures in different files
plot_info = {'y_min': -2.5,
             'y_max': 2.5,
             'subplot_layout_row': 5,
             'subplot_layout_col': 2,
             'title': 'Time series - window '}
number_of_windows = 7
ind_start = 0
incr = plot_info['subplot_layout_row'] * plot_info['subplot_layout_col']
ind_stop = incr
for window_n in range(number_of_windows):
    fig, axis = plt.subplots(plot_info['subplot_layout_row'], plot_info['subplot_layout_col'], figsize=(15, 10))
    fig.suptitle(plot_info['title'] + str(window_n + 1), fontsize=16, y=0.98)
    plt.tight_layout(pad=3.)
    for ax, data_i in zip(axis.flat, data_processed[ind_start:ind_stop]):
        for i in range(data_i[1].shape[0]):
            ax.plot([t * t_conversion for t in range(len(data_i[1][i, :]))], data_i[1][i, :],
                    color=colors[i], alpha=0.5)
        ax.set_title(data_i[0], y=1.5, pad=-14)
        ax.set_xlabel('Time [s]')
        ax.set_ylabel('y [\u03BCm]')
        ax.set_ylim(plot_info['y_min'], plot_info['y_max'])
        ax.axhline(y=t_weak, color='g', linestyle='--', label='std')
        ax.axhline(y=-t_weak, color='g', linestyle='--', label='std')
    ind_start = ind_stop
    ind_stop += incr
    fig.savefig(figure_path + plot_info['title'] + str(window_n + 1))
    plt.close(fig)


In [6]:
#@markdown ### Plot a specific time series
t_conversion = 0.097
#@markdown If you want to focus only on specific signals and compare them, check the following box and enter the names of the signals you are interested in.
#@markdown You can visualize up to 4 signals at the same time, customize the rows and colums.

signal_1 = 'P1365_SMA_BRAN_d58_O2_b_2' #@param {type:"string"}
signal_2 = 'P1365_SMA_RISD_d54_O1_a_2' #@param {type:"string"}
signal_3 = 'P1365_SMA_RISD_d54_O3_b_3' #@param {type:"string"}
signal_4 = '' #@param {type:"string"}

rows = 1 #@param {type:"integer"}
columns = 3 #@param {type:"integer"}

plot_info = {'y_min': -2.5,
             'y_max': 2.5,
             'subplot_layout_row': 34,
             'subplot_layout_col': 2,
             'title': 'All Time Series'}
colors = ["#43C6DB", "#93FFE8", "dodgerblue", "mediumseagreen", "slateblue", "#737CA1", "#29465B", "#368BC1",
          "#AFDCEC", "#66CDAA"]
signal_list = [signal_1, signal_2, signal_3, signal_4]
signal_list_plot = [s for s in signal_list if s]
ind_list = [filename_list.index(s) for s in signal_list_plot]
data_processed_plot = [data_processed[ind] for ind in ind_list]
ind_plot = rows * columns
data_processed_plot = data_processed_plot[:(ind_plot)]

#@markdown If you want to save the figure, check the following box insert, and choose a name.
#@markdown The figure in png and html formats will be saved in the 'figures' folder.
#@markdown (The html formal allows you to have an interactive figure. To open it you have to download it first, then double click on the file and a window with the figure will open in your browser).
save = False #@param {type:"boolean"}
plot_title = '' #@param {type:"string"}
fig_new = make_subplots(rows=rows, cols=columns, subplot_titles=signal_list)
r = 0
c = 0
for i in range(len(data_processed_plot)): #len(data_processed)):

  fig_ = px.line(x=[t * t_conversion for t in range(len(data_processed_plot[i][1][0]))], y=[data_i for data_i in data_processed_plot[i][1]], color_discrete_sequence=colors,
                  range_y=[plot_info['y_min'], plot_info['y_max']])
  fig_.update_layout(showlegend=False)
  fig_.update_traces(opacity=0.5)

  if columns == 1:
    c = 1
    r += 1
  elif rows == 1:
    r = 1
    c += 1
  else:
    if c == 2:
      if r == 0:
        r = 1
      r +=1
      c = 1
    else:
      if r == 0:
        r = 1
      c +=1

  for trace in fig_["data"]:
    fig_new.add_trace(trace, row=r, col=c)
    fig_new.update_xaxes(title_text='Time [s]', row=r, col=c)
    fig_new.update_yaxes(title_text='y [\u03BCm]', row=r, col=c, range=[plot_info['y_min'], plot_info['y_max']])
fig_new.update_layout(height=800, width=1500)
fig_new.update_layout(showlegend=False)

if save:
  if plot_title == '':
    print('Attention! If you want to save the figure, please choose a name, and execute the cell!')
  else:
    fig_new.write_html(figure_path + plot_title + ".html")
    fig_new.write_image(figure_path + plot_title + ".png", format='png')
fig_new.show()

In [7]:
#@markdown ## Create DataFrame with extracted features and data info
#@markdown Running this cell you will extract the features and create a table with all the data info and the extracted features, and update the excel sheet.

#@markdown - Energy
#@markdown - Mean
#@markdown - Percentage of Count Above Threshold
#@markdown - Percentage of Absolute Sum of Changes
#@markdown - 75 Quantile
#@markdown - Standard Deviazion

#@markdown Here you can set the threshold for the feature *Percentage of Count Above Threshold*. The default value is 0.5.
threshold = 0.5 #@param
# Feature extraction
df_data = create_feature_in_dataframe(data_processed, threshold)
#Create organoid column
data_info['Organoid'] = data_info['Video name'].apply(lambda x: os.path.basename(x).split("_")[-3])
#Create position column
data_info['Position'] = data_info['Video name'].apply(lambda x: os.path.basename(x).split("_")[-2])

# Set index to merge the two dataframe, updateing the values that are computed from tfresh
data_info.set_index('Video name', inplace=True)
df_data.set_index('Video name', inplace=True)
data_info.update(df_data)
data_info.reset_index(drop=False, inplace=True)

#@markdown ## Choose name for csv export
#@markdown You can save the same table with all the information on the info data and the extracted features in csv file
csv_filename = 'Datasheet_template_example.csv' #@param {type:"string"}
data_info.to_csv(csv_filename) #Here was data_final


## Univariate analysis

In [8]:
#@markdown ### Extracted Time Features Boxplots
#@markdown Choose the feature for the classification and plot the boxplot for the different variables.
fig_uni_variate = make_subplots(rows=2, cols=3)
ax_list = [[1, 1], [1, 2], [1, 3], [2, 1], [2, 2], [2, 3]]
ind = 0

x_boxplot = 'Treatment during contraction' #@param ['Treatment during contraction', 'Phenotype']

#@markdown The figure will be automatically saved in the 'figures' folder.
extracted_feature_columns = ['Energy', 'Mean',
       '%_of_count_above_threshold', '%_of_absolute_sum_of_changes',
       'Quantile_75', 'Standard_deviation', '%_of_count_above_mean']
for ax, col in zip(ax_list, extracted_feature_columns):
    fig_ = px.box(data_info, x=x_boxplot, y=col, color=x_boxplot)# , points="outliers") # Options: None, all, ouliers, suspectedoutliers")
    for trace in fig_["data"]:
        trace.legendgroup = trace.name
        fig_uni_variate.add_trace(trace, row=ax[0], col=ax[1])
        if ind != 0:
            fig_uni_variate.data[-1].showlegend = False

    if col == 'Energy':
        units = 'Normalized Energy [\u03BCm^2 / s]'
    elif col in ['Standard_deviation', 'Mean', 'Quantile_75']:
        units = col + ' [\u03BCm]'
    else:
        units = col
    fig_uni_variate.update_layout(height=800, width=1500)
    fig_uni_variate.update_xaxes(title_text=x_boxplot, row=ax[0], col=ax[1])
    fig_uni_variate.update_yaxes(title_text=units, row=ax[0], col=ax[1])
    ind += 1

fig_uni_variate.show()
fig_uni_variate.write_html(figure_path + "Univariate - boxplot - " + x_boxplot +".html")
fig_uni_variate.write_image(figure_path + "Univariate - boxplot - " + x_boxplot + ".png", format='png')


In [10]:
#@markdown ## Customize colors and markers
#@markdown Run the cell and then choose the colors and markers for the plots.

display_markdown(''' Select a marker for each **Position**:''', raw=True)

# Marker
categories_phenotype = data_info['Position'].unique()
default_marker = 'circle'
dict_markers = {}

def update_marker(change):
    category = change.owner.description[:-9]
    new_marker = change.new
    dict_markers[category] = new_marker

for c in categories_phenotype:
    w = widgets.Dropdown(
        options=['triangle-up', 'circle', 'diamond', 'cross', 'square', 'circle-open', 'square-open'],
        description=' category '+str(c) ,
        value=default_marker,
        disabled=False
    )
    w.observe(update_marker, names='value')
    display(w)

  # Check if category already exists in dictionary
    if c not in dict_markers:
      dict_markers[c] = default_marker

# Color
display_markdown(''' Select a color for each **Organoid**:''', raw=True)
organoids = data_info['Organoid'].unique()
organoid_color_map = {}
default_color = 'green'
for organoid in organoids:
    organoid_color_map[organoid] = default_color

color_dropdowns = {}
def update_color(change):
    organoid = change.owner.description.split()[1]
    new_color = change.new
    organoid_color_map[organoid] = new_color

for organoid in organoids:
    color_picker = widgets.ColorPicker(
        description=f'Organoid {organoid}',
        value=default_color,
    )
    color_picker.observe(update_color, names='value')
    display(color_picker)


 Select a marker for each **Position**:

Dropdown(description=' category a', index=1, options=('triangle-up', 'circle', 'diamond', 'cross', 'square', '…

Dropdown(description=' category b', index=1, options=('triangle-up', 'circle', 'diamond', 'cross', 'square', '…

Dropdown(description=' category c', index=1, options=('triangle-up', 'circle', 'diamond', 'cross', 'square', '…

 Select a color for each **Organoid**:

ColorPicker(value='green', description='Organoid O1')

ColorPicker(value='green', description='Organoid O2')

ColorPicker(value='green', description='Organoid O3')

ColorPicker(value='green', description='Organoid O4')

In [28]:
#@markdown ### Violin + Scatter Plot
#@markdown Choose the feature for the classification and plot the boxplot for the different variables.

ind = 0

x_violin_plot = 'Treatment during contraction' #@param ['Treatment during contraction', 'Phenotype']
y_violin_plot = 'Energy' #@param ["Energy", "%_of_absolute_sum_of_changes", "%_of_count_above_threshold", "Quantile_75", "Standard_deviation", "Total Pixel Count"]
color_violin = 'Treatment during contraction' #@param ['Organoid', 'Position', 'Treatment during contraction', 'Phenotype']

treatment_color_map = {'None': 'lightgrey', 'BRAN': 'lightgrey', 'RISD': 'lightgrey'}


fig = px.violin(data_info, x=x_violin_plot, y=y_violin_plot,
                      color=color_violin,
                      color_discrete_map=treatment_color_map,
                      hover_data=data_info.columns
  )

fig.update_layout(height=700, width=1000)
# unique_legends = []
fig.update_traces(
            showlegend=False,
        )
# for trace in fig["data"]:
#     trace.legendgroup = trace.name
#     position = trace.name
#     trace.alignmentgroup = False
#     #color = organoid_color_map.get(position.split(',')[0])
#     if y_violin_plot == 'Energy':
#         units = 'Normalized Energy [\u03BCm^2 / s]'
#     elif y_violin_plot in ['Standard_deviation', 'Mean', 'Quantile_75']:
#         units = col + ' [\u03BCm]'
#     elif y_violin_plot == 'Total Pixel Count':
#         units = y_violin_plot + " [pxl]"
#     else:
#         units = y_violin_plot

# Scatter
fig_scatter = px.scatter(data_info, x=x_violin_plot, y=y_violin_plot,
                 color='Organoid',
                 color_discrete_map=organoid_color_map,
                 symbol='Position',
                 hover_name='Video name',
                 #color_discrete_sequence=color_scheme#px.colors.qualitative.Alphabet
                 )

unique_legends = set()

for trace in fig_scatter.data:
    position = trace.name
    marker_symbol = dict_markers.get(position.split(',')[1][1:], 'circle')
    legend_name = f"{position.split(',')[1][1:]} - {trace.hovertext[0]}"
    treatment = trace.hovertext[0]
    #color = organoid_color_map.get(position.split(',')[0], default_color)

    fig_scatter.update_traces(
      marker=dict(symbol=marker_symbol, size=7),
      selector=dict(name=position),
    )
    if legend_name not in unique_legends:
        unique_legends.add(legend_name)
    else:
        fig_scatter.update_traces(
            showlegend=False,
        )

for trace in fig_scatter.data:
  fig.add_trace(trace)

fig.update_layout(scattermode="group", scattergap=0.95) # to be considered?
fig.update_layout(height=800, width=1500)
fig.update_xaxes(title_text=x_violin_plot)
fig.update_yaxes(title_text=units)

#@markdown The figure will be automatically saved in the 'figures' folder.
fig.show()
fig.write_html(figure_path + "Univariate_violin_scatter_" + x_violin_plot +".html")
fig.write_image(figure_path + "Univariate_violin_scatter_ " + x_violin_plot + ".png", format='png')


## Bi-variate analysis

In [19]:
#@markdown ## Customize the plot
#@markdown Run the cell and then choose the colors and markers for the plots.
display_markdown(''' Select a color for each **Treatment**:''', raw=True)
categories_treatment = data_info['Treatment during contraction'].unique()
default_color = 'blue'
dict_colors = {}

def update_color(change):
    category = change.owner.description[:-9]
    new_color = change.new if change.new else default_color
    dict_colors[category] = new_color

for c in categories_treatment:
    w = widgets.ColorPicker(
        concise=False,
        description=str(c) + ' category',
        value=default_color,
        disabled=False
    )
    w.observe(update_color, names='value')
    display(w)

    # Check if category already exists in dictionary
    if c not in dict_colors:
        dict_colors[c] = default_color

display_markdown(''' Select a marker for each **Phenotype**:''', raw=True)
categories_phenotype = data_info['Phenotype'].unique()
default_marker = 'circle'
dict_markers = {}

def update_marker(change):
    category = change.owner.description[:-9]
    new_marker = change.new
    dict_markers[category] = new_marker

for c in categories_phenotype:
    w = widgets.Dropdown(
        options=['circle', 'diamond', 'cross', 'square', 'circle-open', 'square-open'],
        description=str(c) + ' category',
        value=default_marker,
        disabled=False
    )
    w.observe(update_marker, names='value')
    display(w)

  # Check if category already exists in dictionary
    if c not in dict_markers:
      dict_markers[c] = default_marker

 Select a color for each **Treatment**:

ColorPicker(value='blue', description='None category')

ColorPicker(value='blue', description='BRAN category')

ColorPicker(value='blue', description='RISD category')

 Select a marker for each **Phenotype**:

Dropdown(description='control category', options=('circle', 'diamond', 'cross', 'square', 'circle-open', 'squa…

Dropdown(description='SMA category', options=('circle', 'diamond', 'cross', 'square', 'circle-open', 'square-o…

In [20]:
dict_colors

{'None': 'blue', 'BRAN': 'red', 'RISD': 'green'}

In [21]:
dict_markers

{'control': 'circle', 'SMA': 'square'}

In [22]:
#@markdown ## Create bivariate plot
#@markdown Run thi cell to create the plots and save them directly in the figures folder
var_plot = [
    ['Energy', '%_of_absolute_sum_of_changes', 1, 1],
    ['Energy', 'Quantile_75', 1, 2],
    ['%_of_count_above_threshold', '%_of_absolute_sum_of_changes', 1, 3],
    ['%_of_absolute_sum_of_changes', 'Quantile_75', 2, 1],
    ['%_of_absolute_sum_of_changes', 'Standard_deviation', 2, 2],
    ['Quantile_75', 'Standard_deviation', 2, 3]
]

fig_final = make_subplots(rows=2, cols=3)

ind = 0
unique_legends = set()

for x, y, r, c in var_plot:
    fig_ = px.scatter(data_info, x=x, y=y,
                      color='Treatment during contraction',
                      color_discrete_map=dict_colors,
                      symbol='Phenotype',
                      hover_name='Video name'
                      )

    for trace in fig_.data:
        phenotype = trace.name
        split_phenotype = phenotype.split(',')
        if len(split_phenotype) > 1:
          marker_symbol = dict_markers.get(split_phenotype[1][1:], 'circle')
          legend_name = f"{phenotype.split(',')[1][1:]} - {trace.hovertext[0]}"
        else:
          # Handle the case where there aren't enough elements after splitting
          # For example:
          marker_symbol = 'default_value'
          legend_name = 'deafault_for_this_as_well'
        #marker_symbol = dict_markers.get(phenotype.split(',')[1][1:], 'circle')
        # legend_name = f"{phenotype.split(',')[1][1:]} - {trace.hovertext[0]}"
        treatment = trace.hovertext[0]
        color= dict_colors.get(phenotype.split(',')[0])

        if legend_name not in unique_legends:
            fig_final.add_trace(
                go.Scatter(
                    x=trace.x,
                    y=trace.y,
                    mode=trace.mode,
                    name=trace.name,
                    legendgroup=trace.legendgroup,
                    marker=dict(symbol=marker_symbol, color=color),  # Assign color based on treatment
                    hovertext=trace.hovertext
                ),
                row=r, col=c
            )
            unique_legends.add(legend_name)
        else:
            fig_final.add_trace(
                go.Scatter(
                    x=trace.x,
                    y=trace.y,
                    mode=trace.mode,
                    name=trace.name,
                    legendgroup=trace.legendgroup,
                    showlegend=False,
                    marker=dict(symbol=marker_symbol, color=color),  # Assign color based on treatment
                    hovertext=trace.hovertext
                ),
                row=r, col=c
            )

    fig_final.update_xaxes(title_text=x, row=r, col=c)
    fig_final.update_yaxes(title_text=y, row=r, col=c)

fig_final.update_layout(height=800, width=1500)
fig_final.write_html(figure_path + "Bivariate.html")
fig_final.write_image(figure_path + "Bivariate.png")
fig_final.show()

In [23]:
#@markdown ## Focus on one plot
#@markdown Choose the quantities that you want to plot to have an interactive plot where you can zoom in and out and change colors figure you want to analyse better

x = '%_of_count_above_threshold' #@param ["Energy", "%_of_absolute_sum_of_changes", "%_of_count_above_threshold", "Quantile_75", "Standard_deviation"]
y = '%_of_absolute_sum_of_changes' #@param ["Energy", "%_of_absolute_sum_of_changes", "%_of_count_above_threshold", "Quantile_75", "Standard_deviation"]

fig = px.scatter(data_info, x=x, y=y,
                 color='Treatment during contraction',
                 color_discrete_map=dict_colors,
                 symbol='Phenotype',
                 hover_name='Video name'
                 )

# unique_legends = set()

# for trace in fig.data:
#     phenotype = trace.name
#     marker_symbol = dict_markers.get(phenotype.split(',')[1][1:], 'circle')
#     legend_name = f"{phenotype.split(',')[1][1:]} - {trace.hovertext[0]}"
#     treatment = trace.hovertext[0]
#     color = dict_colors.get(phenotype.split(',')[0])
#     print("HERE", phenotype, color)

#     if legend_name not in unique_legends:
#         fig.update_traces(
#             marker=dict(symbol=marker_symbol, color=color),  # Assign marker and color based on treatment
#             selector=dict(name=trace.name)
#         )
#         unique_legends.add(legend_name)
#     else:
#         fig.update_traces(
#             showlegend=False,
#             marker=dict(symbol=marker_symbol, color=color),  # Assign marker and color based on treatment
#             selector=dict(name=trace.name)
#         )

# fig.update_layout(height=800, width=1500)
# fig.write_html(figure_path + "Bivariate.html")
# fig.write_image(figure_path + "Bivariate.png")

# marker_size = 6 #@param {type:"slider", min:1, max:10, step:1}
# fig.update_traces(marker_size=marker_size)

# #@markdown If you want to save the figure, check the following box insert, and choose a name.
# save = False #@param {type:"boolean"}
# figure_name = "Plot_1" #@param {type:"string"}
# if save:
#   if figure_name == '':
#     print('Attention! If you want to save the figure, please choose a name, and execute the cell!')
#   else:
#     fig.write_image(figure_path + figure_name + ".png")
#     fig.write_html(figure_path + figure_name + ".html")
fig.show()

In [29]:
#@markdown ## Plot Total Count Muscle and Neural Parts - Second Try - Overlap
#@markdown This visualization illustrates the cumulative counts (which will represent area in the future) for each image/organoid
#@markdown along with their corresponding ratios, distinguishing between the neural and muscle components.
#@markdown Colors are assigned based on the previously defined organoid_color_map,
#@markdown with the muscle component represented by a lighter shade and the neural component by a darker hue.

# Filter the data and keep only unique Image names
data_info_reduced = data_info.drop_duplicates(subset=['Image name']).copy()

# Convert the ratio into counts to plot the net counts and not the ratios
data_info_reduced['Neural Count'] = data_info_reduced['Neural Ratio'] * data_info_reduced['Total Pixel Count']
data_info_reduced['Muscle Count'] = data_info_reduced['Muscle Ratio'] * data_info_reduced['Total Pixel Count']


# Plot Muscular part
fig_muscle = px.bar(data_info_reduced,
                    x="Image name",
                    y="Muscle Count",
                    color="Organoid",
                    facet_col="Treatment during contraction",
                    facet_col_wrap=3,
                    color_discrete_map=organoid_color_map,
                    title="Muscular Part",
                   )
fig_muscle.update_traces(showlegend=False)

# Plot Neural part with darker shade
fig_neural = px.bar(data_info_reduced,
                    x="Image name",
                    y="Neural Count",
                    color="Organoid",
                    facet_col="Treatment during contraction",
                    facet_col_wrap=3,
                    color_discrete_map=organoid_color_map,
                    title="Neural Part",
                   )
fig_neural.update_traces(showlegend=False)

legend_items = []
for organoid, color in organoid_color_map.items():
    legend_items.append(go.Scatter(x=[None], y=[None], mode='markers',
                                   marker=dict(symbol='square', color=color, size=10, opacity=0.5),
                                   showlegend=True,
                                   legendgroup=organoid,
                                   name=f"{organoid} - Muscle"))

for organoid, color in organoid_color_map.items():
    legend_items.append(go.Scatter(x=[None], y=[None], mode='markers',
                                   marker=dict(symbol='square', color=color, size=10),
                                   showlegend=True,
                                   legendgroup=organoid,
                                   name=f"{organoid} - Neural"))

# Combine the two plots
fig_combined = fig_muscle.update_traces(marker=dict(opacity=0.5))
fig_combined.add_traces(fig_neural.data)
fig_combined.add_traces(legend_items)
fig_combined.update_layout(legend_title_text='Legend')
fig_combined.update_layout(yaxis_title="Total Pixel Count")
fig_combined.show()

#Save figures in image folder
fig_combined.write_html(figure_path + "Total_counts_and_ratios.html")
fig_combined.write_image(figure_path + "Total_counts_and_ratios.png", format='png')


---